# Scenario 2 – Item-Based Collaborative Filtering
## Movie Recommendation System using Item Similarity
### Dataset: MovieLens 100K

---
## Step 1: Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import networkx as nx

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

---
## Step 2: Load Dataset

> **Setup:** Download `ml-100k.zip` from https://grouplens.org/datasets/movielens/100k/  
> Unzip and place the `ml-100k/` folder in the same directory as this notebook.

In [ ]:
DATA_PATH = 'ml-100k'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Dataset folder 'ml-100k' not found. "
        "Download from https://grouplens.org/datasets/movielens/100k/ "
        "and place the unzipped folder next to this notebook."
    )

ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
movies_cols  = ['movie_id', 'title', 'release_date', 'video_release_date',
                'IMDb_URL'] + [f'genre_{i}' for i in range(19)]

ratings = pd.read_csv(f'{DATA_PATH}/u.data', sep='\t',
                      names=ratings_cols, encoding='latin-1')
movies  = pd.read_csv(f'{DATA_PATH}/u.item', sep='|',
                      names=movies_cols,  encoding='latin-1')

df = ratings.merge(movies[['movie_id', 'title']], on='movie_id', how='left')

print(f'Ratings shape : {ratings.shape}')
print(f'Movies shape  : {movies.shape}')
print(f'Merged shape  : {df.shape}')
df.head()

---
## Step 3: Create Item-User Matrix

In [ ]:
# Rows = movies, Columns = users  (transpose of user-item matrix)
item_user_matrix = df.pivot_table(index='title', columns='user_id',
                                   values='rating')

print('Item-User matrix shape:', item_user_matrix.shape)

# Sparsity
total   = item_user_matrix.shape[0] * item_user_matrix.shape[1]
missing = item_user_matrix.isnull().sum().sum()
sparsity_item = (missing / total) * 100
print(f'Sparsity: {sparsity_item:.2f}%')

# Fill NaN with 0 for similarity computation
item_user_filled = item_user_matrix.fillna(0)
item_user_matrix.iloc[:5, :8]

---
## Step 4: Compute Item Similarity – Cosine & Pearson

In [ ]:
# --- 4a: Cosine Similarity ---
cosine_sim_values = cosine_similarity(item_user_filled)
item_cosine_sim   = pd.DataFrame(cosine_sim_values,
                                  index=item_user_matrix.index,
                                  columns=item_user_matrix.index)
print('Cosine similarity matrix shape:', item_cosine_sim.shape)

# --- 4b: Pearson Correlation ---
# pandas .T.corr() gives item-item Pearson correlation
item_pearson_sim = item_user_matrix.T.corr(method='pearson')
item_pearson_sim.fillna(0, inplace=True)
print('Pearson similarity matrix shape:', item_pearson_sim.shape)

print('\nSample Cosine Similarities (first 4x4):')
item_cosine_sim.iloc[:4, :4]

---
## Step 5: Identify Top Similar Items

In [ ]:
def get_similar_items(item_title, similarity_df, n=10):
    """
    Return top-N items most similar to item_title.
    Excludes the item itself.
    """
    if item_title not in similarity_df.index:
        raise ValueError(f"'{item_title}' not found in similarity matrix.")
    sim_scores = similarity_df[item_title].drop(index=item_title)
    return sim_scores.nlargest(n)

# Example: find movies similar to Star Wars (1977)
QUERY_MOVIE = 'Star Wars (1977)'
similar_cosine  = get_similar_items(QUERY_MOVIE, item_cosine_sim,  n=10)
similar_pearson = get_similar_items(QUERY_MOVIE, item_pearson_sim, n=10)

print(f'Top 10 items similar to "{QUERY_MOVIE}" (Cosine):')
print(similar_cosine.to_frame('Cosine Similarity').to_string())
print(f'\nTop 10 items similar to "{QUERY_MOVIE}" (Pearson):')
print(similar_pearson.to_frame('Pearson Correlation').to_string())

---
## Step 6: Recommend Items Based on User History

In [ ]:
def item_based_recommend(user_id, df, similarity_df, n_recommendations=10,
                          n_similar=5, min_rating=4.0):
    """
    For a given user:
    1. Find movies they rated highly (>= min_rating)
    2. For each liked movie, find top-N similar movies
    3. Aggregate scores, filter already-seen movies
    4. Return top recommendations
    """
    # Movies the user has rated highly
    liked = (df[(df['user_id'] == user_id) & (df['rating'] >= min_rating)]
               ['title'].tolist())
    all_seen = df[df['user_id'] == user_id]['title'].tolist()

    if not liked:
        return pd.Series(dtype=float, name='Score')

    scores = {}
    for movie in liked:
        if movie not in similarity_df.index:
            continue
        sim_movies = get_similar_items(movie, similarity_df, n=n_similar)
        for title, score in sim_movies.items():
            if title not in all_seen:
                scores[title] = scores.get(title, 0) + score

    rec_series = pd.Series(scores).sort_values(ascending=False)
    return rec_series.head(n_recommendations)

TARGET_USER = 1
recs = item_based_recommend(TARGET_USER, df, item_cosine_sim,
                             n_recommendations=10)

print(f'Top 10 Item-Based Recommendations for User {TARGET_USER}:')
print('=' * 55)
for rank, (movie, score) in enumerate(recs.items(), 1):
    print(f'  {rank:2}. {movie[:45]:<45}  (score: {score:.4f})')

---
## Step 7: Compare Item-Based vs User-Based Recommendations

In [ ]:
# ---- Rebuild user-based recommendations (from Scenario 1) ----
user_item_matrix = df.pivot_table(index='user_id', columns='title',
                                   values='rating')
user_item_filled = user_item_matrix.fillna(0)
user_sim_values  = cosine_similarity(user_item_filled)
user_sim_df      = pd.DataFrame(user_sim_values,
                                 index=user_item_matrix.index,
                                 columns=user_item_matrix.index)

def get_top_n_similar_users(user_id, similarity_df, n=10):
    return similarity_df[user_id].drop(index=user_id).nlargest(n)

def predict_user_ratings(user_id, user_item_df, similarity_df, n_neighbors=10):
    top_users        = get_top_n_similar_users(user_id, similarity_df, n=n_neighbors)
    sim_weights      = top_users.values
    neighbor_ratings = user_item_df.loc[top_users.index]
    rated_movies     = user_item_df.loc[user_id].dropna().index
    weighted_sum     = neighbor_ratings.T.dot(sim_weights)
    sim_total        = np.where(neighbor_ratings.T.notna().dot(sim_weights) == 0,
                                1e-9, neighbor_ratings.T.notna().dot(sim_weights))
    predicted        = pd.Series(weighted_sum / sim_total, index=user_item_df.columns)
    return predicted.drop(index=rated_movies, errors='ignore').sort_values(ascending=False)

user_recs = predict_user_ratings(TARGET_USER, user_item_matrix, user_sim_df)
user_top10 = user_recs.head(10)

# --- Comparison ---
item_set = set(recs.index)
user_set = set(user_top10.index)
overlap  = item_set & user_set

print(f'User {TARGET_USER} – Recommendation Overlap')
print(f'  Item-Based top-10  : {len(item_set)} movies')
print(f'  User-Based top-10  : {len(user_set)} movies')
print(f'  Common movies      : {len(overlap)}')
print(f'  Overlap %          : {len(overlap)/10*100:.0f}%')
if overlap:
    print(f'  Shared movies      : {list(overlap)}')

---
## Step 8: Evaluate – RMSE & Precision@K

In [ ]:
# Train/test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Build item-user matrix from training data
train_item_user  = train_df.pivot_table(index='title', columns='user_id',
                                         values='rating').fillna(0)
train_item_cosim = pd.DataFrame(cosine_similarity(train_item_user),
                                 index=train_item_user.index,
                                 columns=train_item_user.index)

# ---- RMSE ----
def predict_item_rating(user_id, movie_title, train_item_user_df,
                         item_sim_df, n_similar=10):
    """
    Predict a single rating for user_id on movie_title using item-based CF.
    """
    if movie_title not in item_sim_df.index:
        return np.nan
    if user_id not in train_item_user_df.columns:
        return np.nan

    sim_movies = item_sim_df[movie_title].drop(index=movie_title).nlargest(n_similar)
    valid      = sim_movies[sim_movies.index.isin(train_item_user_df.index)]
    if valid.empty:
        return np.nan

    user_col    = train_item_user_df.loc[valid.index, user_id]
    rated_mask  = user_col > 0
    if rated_mask.sum() == 0:
        return np.nan

    weighted_sum = (valid[rated_mask] * user_col[rated_mask]).sum()
    sim_sum      = valid[rated_mask].abs().sum()
    return weighted_sum / sim_sum if sim_sum != 0 else np.nan

actual_rmse, predicted_rmse = [], []
test_sample = test_df.sample(n=500, random_state=42)

for _, row in test_sample.iterrows():
    pred = predict_item_rating(row['user_id'], row['title'],
                                train_item_user, train_item_cosim)
    if not np.isnan(pred):
        actual_rmse.append(row['rating'])
        predicted_rmse.append(pred)

rmse = np.sqrt(mean_squared_error(actual_rmse, predicted_rmse))
mae  = np.mean(np.abs(np.array(actual_rmse) - np.array(predicted_rmse)))
print(f'RMSE : {rmse:.4f}  (on {len(actual_rmse)} predictions)')
print(f'MAE  : {mae:.4f}')

In [ ]:
# ---- Precision@K ----
def precision_at_k(user_id, df_full, train_df, item_sim_df,
                   k=10, relevance_threshold=4.0):
    """
    Precision@K = (# recommended items that are relevant) / K
    Relevant = rated >= relevance_threshold in the full dataset
    (simulating held-out ground truth)
    """
    recs = item_based_recommend(user_id, train_df, item_sim_df,
                                 n_recommendations=k)
    if recs.empty:
        return 0.0

    relevant = set(df_full[(df_full['user_id'] == user_id) &
                            (df_full['rating'] >= relevance_threshold)]['title'])
    hits = sum(1 for m in recs.index if m in relevant)
    return hits / k

K = 10
sample_users = df['user_id'].unique()[:50]
prec_scores  = [precision_at_k(u, df, train_df, train_item_cosim, k=K)
                for u in sample_users]

mean_prec = np.mean(prec_scores)
print(f'Precision@{K} (mean over {len(sample_users)} users): {mean_prec:.4f}')
print(f'  = {mean_prec*100:.1f}% of top-{K} recommendations are relevant')

---
## Analysis Tasks
### Popular vs Niche Items & Scalability

In [ ]:
# Ratings count per movie
rating_counts = df.groupby('title')['rating'].agg(['count', 'mean']).reset_index()
rating_counts.columns = ['title', 'num_ratings', 'avg_rating']
rating_counts = rating_counts.sort_values('num_ratings', ascending=False)

popular_threshold = rating_counts['num_ratings'].quantile(0.75)
niche_threshold   = rating_counts['num_ratings'].quantile(0.25)

popular_movies = rating_counts[rating_counts['num_ratings'] >= popular_threshold]
niche_movies   = rating_counts[rating_counts['num_ratings'] <= niche_threshold]

print(f'Popular items (>= {popular_threshold:.0f} ratings) : {len(popular_movies)}')
print(f'Niche items   (<= {niche_threshold:.0f} ratings)  : {len(niche_movies)}')
print(f'\nTop 10 Most Rated Movies:')
print(rating_counts.head(10)[['title','num_ratings','avg_rating']].to_string(index=False))

# Scalability note
n_items = item_user_matrix.shape[0]
n_users = item_user_matrix.shape[1]
print(f'\nScalability Analysis:')
print(f'  Items : {n_items}  |  Users : {n_users}')
print(f'  Item similarity matrix size : {n_items}×{n_items} = {n_items**2:,} cells')
print(f'  User similarity matrix size : {n_users}×{n_users} = {n_users**2:,} cells')
print(f'  Item-based is more stable: item count grows slower than user count.')

---
## Visualizations
### Visualization 1 – Item Similarity Heatmap

In [ ]:
# Use top 25 most-rated movies for a readable heatmap
top25 = rating_counts.head(25)['title'].tolist()
sim_subset = item_cosine_sim.loc[top25, top25]

plt.figure(figsize=(14, 12))
mask = np.eye(len(top25), dtype=bool)   # mask the diagonal (self-similarity = 1)
sns.heatmap(sim_subset, cmap='magma', annot=False, mask=mask,
            linewidths=0.3, linecolor='#333',
            cbar_kws={'label': 'Cosine Similarity'},
            xticklabels=[t[:22] for t in top25],
            yticklabels=[t[:22] for t in top25])
plt.title('Item-Item Cosine Similarity Heatmap\n(Top 25 Most-Rated Movies)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

### Visualization 2 – Top Similar Items Graph (Network)

In [ ]:
# Build a network graph: query movie + its top similar items
GRAPH_MOVIE = 'Star Wars (1977)'
n_neighbors = 8
similar_nodes = get_similar_items(GRAPH_MOVIE, item_cosine_sim, n=n_neighbors)

G = nx.Graph()
G.add_node(GRAPH_MOVIE, node_type='query')
for movie, score in similar_nodes.items():
    G.add_node(movie, node_type='similar')
    G.add_edge(GRAPH_MOVIE, movie, weight=score)

# Also add 2nd-level neighbours for depth
for movie in list(similar_nodes.index)[:4]:
    lvl2 = get_similar_items(movie, item_cosine_sim, n=3)
    for m2, s2 in lvl2.items():
        if m2 not in G.nodes:
            G.add_node(m2, node_type='level2')
        G.add_edge(movie, m2, weight=s2)

pos = nx.spring_layout(G, seed=42, k=2.0)
node_colors = ['#e63946' if G.nodes[n]['node_type'] == 'query'
               else '#457b9d' if G.nodes[n]['node_type'] == 'similar'
               else '#a8dadc' for n in G.nodes]
node_sizes  = [3000 if G.nodes[n]['node_type'] == 'query'
               else 1200 if G.nodes[n]['node_type'] == 'similar'
               else 600 for n in G.nodes]
edge_weights = [G[u][v]['weight'] * 3 for u, v in G.edges]

plt.figure(figsize=(14, 9))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9)
nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.5, edge_color='#888')
labels = {n: n[:20] for n in G.nodes}
nx.draw_networkx_labels(G, pos, labels, font_size=7, font_color='white', font_weight='bold')
plt.title(f'Similar Movies Network: "{GRAPH_MOVIE}"\n'
          f'Red = Query | Blue = Direct Neighbours | Teal = 2nd-Level',
          fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

### Visualization 3 – Recommendation Comparison: Item-Based vs User-Based

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ---- Item-Based ----
item_top10 = item_based_recommend(TARGET_USER, df, item_cosine_sim, n_recommendations=10)
colors_item = cm.plasma(np.linspace(0.2, 0.85, len(item_top10)))
axes[0].barh([t[:35] for t in item_top10.index[::-1]],
             item_top10.values[::-1], color=colors_item[::-1], edgecolor='white')
for bar, val in zip(axes[0].patches, item_top10.values[::-1]):
    axes[0].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=8)
axes[0].set_title(f'Item-Based Recommendations\n(User {TARGET_USER})', fontsize=12)
axes[0].set_xlabel('Aggregated Similarity Score')
axes[0].set_xlim(0, item_top10.max() * 1.18)

# ---- User-Based ----
user_top10_plot = user_recs.head(10)
colors_user = cm.viridis(np.linspace(0.2, 0.85, len(user_top10_plot)))
axes[1].barh([t[:35] for t in user_top10_plot.index[::-1]],
             user_top10_plot.values[::-1], color=colors_user[::-1], edgecolor='white')
for bar, val in zip(axes[1].patches, user_top10_plot.values[::-1]):
    axes[1].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=8)
axes[1].set_title(f'User-Based Recommendations\n(User {TARGET_USER})', fontsize=12)
axes[1].set_xlabel('Weighted Predicted Rating')
axes[1].set_xlim(0, user_top10_plot.max() * 1.18)

plt.suptitle('Item-Based vs User-Based Collaborative Filtering — Top 10 Recommendations',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Overlap annotation
print(f'\nOverlap between both approaches: {len(overlap)} / 10 movies in common')
print(f'Common movies: {list(overlap) if overlap else "None"}')

### Visualization 4 – Popular vs Niche Items & Cosine vs Pearson Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ---- Popular vs Niche ----
categories = ['Popular\n(Top 25%)', 'Mid-Tier\n(Middle 50%)', 'Niche\n(Bottom 25%)']
counts     = [len(popular_movies),
              len(rating_counts) - len(popular_movies) - len(niche_movies),
              len(niche_movies)]
bar_colors = ['#2a9d8f', '#e9c46a', '#e76f51']
axes[0].bar(categories, counts, color=bar_colors, edgecolor='white', width=0.5)
for bar, cnt in zip(axes[0].patches, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(cnt), ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Popular vs Niche Movie Distribution', fontsize=12)
axes[0].set_ylabel('Number of Movies')
axes[0].set_ylim(0, max(counts) * 1.15)

# ---- Cosine vs Pearson: top-10 scores for query movie ----
cos_top10  = get_similar_items(QUERY_MOVIE, item_cosine_sim,  n=10)
pear_top10 = get_similar_items(QUERY_MOVIE, item_pearson_sim, n=10)

x  = np.arange(10)
w  = 0.38
axes[1].bar(x - w/2, cos_top10.values,  width=w, label='Cosine',  color='#264653', alpha=0.85)
axes[1].bar(x + w/2, pear_top10.values, width=w, label='Pearson', color='#e9c46a', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'#{i+1}' for i in range(10)])
axes[1].set_title(f'Cosine vs Pearson Similarity Scores\n(Top 10 for "{QUERY_MOVIE[:25]}...")',
                  fontsize=11)
axes[1].set_ylabel('Similarity Score')
axes[1].legend()

plt.suptitle('Analysis: Item Popularity & Similarity Metric Comparison',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Summary & Observations

In [ ]:
print('=' * 65)
print('  ITEM-BASED COLLABORATIVE FILTERING – SUMMARY')
print('=' * 65)
print(f'  Dataset              : MovieLens 100K')
print(f'  Items (movies)       : {item_user_matrix.shape[0]}')
print(f'  Users                : {item_user_matrix.shape[1]}')
print(f'  Matrix Sparsity      : {sparsity_item:.2f}%')
print(f'  RMSE                 : {rmse:.4f}')
print(f'  MAE                  : {mae:.4f}')
print(f'  Precision@{K}         : {mean_prec:.4f} ({mean_prec*100:.1f}%)')
print(f'  Rec. overlap w/ UB   : {len(overlap)}/10 movies')
print('=' * 65)

print('''
ANALYSIS OBSERVATIONS:

1. SIMILARITY METRICS:
   Cosine similarity handles sparse vectors better and tends
   to give more stable scores. Pearson correlation captures
   rating scale differences between users but is noisier on
   items rated by few users.

2. POPULAR VS NICHE:
   Popular items get high-quality similarity scores because
   many users have co-rated them. Niche items with few ratings
   suffer from sparse overlap, yielding unreliable similarity.

3. ITEM-BASED vs USER-BASED:
   Item-based CF is more stable over time (item catalogues
   change less than user preferences). User-based CF is more
   personalised but requires recomputing similarity as users
   add ratings.

4. SCALABILITY:
   With 943 users and 1682 items, the item similarity matrix
   (1682×1682) is smaller than the user similarity matrix
   (943×943 here, but user bases typically grow much faster).
   Item-based CF precomputes and caches item similarities
   efficiently, making it preferred in production systems.

5. PRECISION@K:
   Measures recommendation usefulness directly — a higher
   Precision@10 means more top-10 suggestions are movies the
   user would actually enjoy.
''')